In [0]:
%sql
-- mq_gmdf_dev.oil_obs.v_llm_bronze
CREATE OR REPLACE VIEW mq_gmdf_dev.oil_obs.v_llm_bronze AS
SELECT
    id, shift_date, shift_type, batch_nbr, capability, scheduler_run,
    model_config, transport, success, error_msg, latency_ms, called_at, ingestion_ts,
    unix_timestamp(ingestion_ts) - unix_timestamp(called_at) AS write_lag_s,
    user_prompt, system_prompt, response_raw, response_parsed,
        CASE
      WHEN model_config = 'demo-claude-sonnet-4-6-pwc-omi'
       AND error_msg    = 'Unable to locate credentials'
        THEN true ELSE false
    END AS is_credential_fastfail,
            CASE
      WHEN success = true
       AND (response_parsed IS NULL
            OR length(trim(cast(response_parsed AS STRING))) = 0
            OR cast(response_parsed AS STRING) IN ('{}', '[]', 'null')
            -- valid JSON with a null how_we_ran: the one field summary's system prompt calls
            -- "always present". This shape read as healthy before.
            OR (capability = 'summary'
                AND coalesce(trim(try_parse_json(cast(response_parsed AS STRING)):how_we_ran::string), '') = ''))
        THEN true ELSE false
    END AS is_blank_output,

        -- error taxonomy: timeouts are latency events, not generic failures
        CASE
      WHEN success = true                                               THEN NULL
      WHEN error_msg ILIKE '%timed out%' OR error_msg ILIKE '%timeout%'  THEN 'timeout'
      WHEN error_msg ILIKE '%credential%'
        OR error_msg ILIKE '%AccessDenied%'
        OR error_msg ILIKE '%403%'
        OR error_msg ILIKE '%Forbidden%'
        OR error_msg ILIKE '%401%'
        OR error_msg ILIKE '%Unauthorized%'                             THEN 'auth'
      WHEN error_msg ILIKE '%Internal Server Error%'
        OR error_msg RLIKE '(?i)cortex\\s+5[0-9]{2}'                    THEN 'upstream_5xx'
      WHEN error_msg ILIKE '%rate limit%' OR error_msg ILIKE '%429%'     THEN 'rate_limit'
      WHEN error_msg ILIKE '%connection%'
        OR error_msg ILIKE '%WinError%'
        OR error_msg ILIKE '%reset by peer%'
        OR error_msg ILIKE '%broken pipe%'
        OR error_msg ILIKE '%ECONNRESET%'                               THEN 'connection'
      ELSE 'other'
    END AS error_class,

    -- payload sizes: leading indicator for the dsa_optimize timeouts (user 37,006 chars)
    length(cast(system_prompt   AS STRING)) AS system_prompt_chars,
    length(cast(user_prompt     AS STRING)) AS user_prompt_chars,
    length(cast(response_parsed AS STRING)) AS response_chars,

    try_parse_json(cast(response_parsed AS STRING)) AS resp_v
FROM mq_gmdf_dev.oil.ptof_primary__ai_llm_audit_log;

In [0]:
%sql
-- mq_gmdf_dev.oil_obs.v_ish_bronze
CREATE OR REPLACE VIEW mq_gmdf_dev.oil_obs.v_ish_bronze AS
WITH j AS (
  SELECT
      id, action, entity_type, entity_id, user_id, user_email, ts, change_summary,
      try_parse_json(after_json)  AS after_v,
      try_parse_json(before_json) AS before_v
  FROM mq_gmdf_dev.oil.ptof_ish_audit
)
SELECT
    id, action, entity_type, entity_id, user_id, user_email, ts, change_summary,
    before_v, after_v,
    CAST(coalesce(
        after_v:shift_date::string,
        after_v:shift_date_key::string
    ) AS DATE) AS shift_date_norm,
    coalesce(
        after_v:shift_type::string,
        after_v:shift_label::string
    ) AS shift_type_norm,
    coalesce(
        after_v:batch_id::string,
        after_v:content.batch_id::string
    ) AS batch_id_norm,
    CASE
      WHEN entity_type = 'HandoverEmail'
       AND after_v:sent::boolean = false
       AND after_v:reason::string LIKE '%EMAIL_ENABLED=false%'
      THEN true ELSE false
    END AS is_email_disabled_gate
FROM j;